# Sesión 3 — Soluciones: Agente LangGraph

Versión con soluciones completas para el instructor.

In [ ]:
%pip install -q \
    langgraph \
    langchain \
    langchain-google-genai \
    langchain-ollama \
    langchain-chroma \
    chromadb \
    "datasets<3.0" \
    pandas

In [ ]:
# ── Configuración ────────────────────────────────────────────────────────────
BACKEND = "gemini"
OLLAMA_MODEL = "qwen3:4b"
OLLAMA_EMBEDDING_MODEL = "nomic-embed-text-v2-moe"
# ─────────────────────────────────────────────────────────────────────────────

if BACKEND == "gemini":
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
    from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
    llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=GOOGLE_API_KEY)
    embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001", google_api_key=GOOGLE_API_KEY)
    print("✓ Backend: Gemini")
elif BACKEND == "ollama":
    from langchain_ollama import ChatOllama, OllamaEmbeddings
    llm = ChatOllama(model=OLLAMA_MODEL)
    embeddings = OllamaEmbeddings(model=OLLAMA_EMBEDDING_MODEL)
    print(f"✓ Backend: Ollama — {OLLAMA_MODEL}")

In [ ]:
from datasets import load_dataset
import pandas as pd
from langchain_core.documents import Document
from langchain_chroma import Chroma

ds = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023",
    "raw_review_Electronics",
    split="full",
    streaming=True,
    trust_remote_code=True,
)
df = pd.DataFrame(ds.take(200))[["title", "text", "rating", "parent_asin"]].dropna(subset=["text"])
df["rating"] = df["rating"].astype(int)

docs = [
    Document(
        page_content=f"{row['title']}\n\n{row['text']}",
        metadata={"rating": row["rating"], "parent_asin": row["parent_asin"]}
    )
    for _, row in df.iterrows()
]

vectorstore = Chroma.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print(f"✓ {vectorstore._collection.count()} documentos indexados")

In [ ]:
from typing import TypedDict

class TicketState(TypedDict):
    ticket: str
    category: str
    urgency: str
    summary: str
    rag_context: str
    draft_response: str
    escalate: bool
    final_response: str

## Solución Ejercicio 1: Nodo `clasificar`

In [ ]:
import json

def clasificar(state: TicketState) -> dict:
    prompt = f"""Analiza el siguiente mensaje de un cliente y responde ÚNICAMENTE con un JSON válido.
Sin texto adicional, sin markdown, sin bloques de código.

Campos requeridos:
- "category": una de ["devolucion", "consulta_tecnica", "queja", "envio", "otro"]
- "urgency": una de ["alta", "media", "baja"]
- "summary": resumen del problema en máximo 15 palabras

Mensaje del cliente:
{state['ticket']}"""

    respuesta = llm.invoke(prompt).content.strip()

    if respuesta.startswith("```"):
        respuesta = respuesta.split("```")[1]
        if respuesta.startswith("json"):
            respuesta = respuesta[4:]

    datos = json.loads(respuesta)
    print(f"  [clasificar] → categoría: {datos['category']}, urgencia: {datos['urgency']}")
    return datos

print("✓ clasificar")

In [ ]:
def consultar_catalogo(state: TicketState) -> dict:
    consulta = f"{state.get('category', '')} {state.get('summary', state['ticket'])}"
    docs = retriever.invoke(consulta)
    contexto = "\n\n---\n\n".join(
        f"[{d.metadata.get('rating', '?')}★] {d.page_content[:350]}"
        for d in docs
    )
    print(f"  [consultar_catalogo] → {len(docs)} reviews")
    return {"rag_context": contexto}

print("✓ consultar_catalogo")

## Solución Ejercicio 2: Nodo `redactar_respuesta`

In [ ]:
def redactar_respuesta(state: TicketState) -> dict:
    prompt = f"""Eres el responsable de atención al cliente de una tienda de electrónica.
Tu tono es profesional, empático y directo.

Ticket del cliente:
{state['ticket']}

Categoría: {state.get('category', 'desconocida')} | Urgencia: {state.get('urgency', 'media')}

Información relevante del catálogo de productos:
{state.get('rag_context', 'Sin información disponible')}

Redacta una respuesta de no más de 3 frases que:
1. Reconozca el problema del cliente
2. Aporte información concreta del catálogo si es relevante
3. Indique el siguiente paso claro"""

    borrador = llm.invoke(prompt).content
    print(f"  [redactar_respuesta] → {len(borrador)} chars")
    return {"draft_response": borrador}

print("✓ redactar_respuesta")

In [ ]:
def evaluar_escalado(state: TicketState) -> dict:
    categorias_criticas = {"devolucion", "queja"}
    escalar = (
        state.get("urgency") == "alta"
        and state.get("category") in categorias_criticas
    )
    print(f"  [evaluar_escalado] → escalar: {escalar}")
    return {"escalate": escalar}

def respuesta_automatica(state: TicketState) -> dict:
    return {"final_response": f"[AUTOMÁTICO]\n\n{state['draft_response']}"}

def escalar_a_humano(state: TicketState) -> dict:
    return {"final_response": (
        f"[ESCALADO]\nCategoría: {state.get('category')} | Urgencia: {state.get('urgency')}\n"
        f"Resumen: {state.get('summary')}\n\nBorrador:\n{state['draft_response']}"
    )}

def decidir_ruta(state: TicketState) -> str:
    return "escalar" if state.get("escalate") else "enviar"

print("✓ Nodos de decisión listos")

## Solución Ejercicio 3: El grafo completo

In [ ]:
from langgraph.graph import StateGraph, END

builder = StateGraph(TicketState)

builder.add_node("clasificar", clasificar)
builder.add_node("consultar_catalogo", consultar_catalogo)
builder.add_node("redactar_respuesta", redactar_respuesta)
builder.add_node("evaluar_escalado", evaluar_escalado)
builder.add_node("respuesta_automatica", respuesta_automatica)
builder.add_node("escalar_a_humano", escalar_a_humano)

builder.set_entry_point("clasificar")
builder.add_edge("clasificar", "consultar_catalogo")
builder.add_edge("consultar_catalogo", "redactar_respuesta")
builder.add_edge("redactar_respuesta", "evaluar_escalado")
builder.add_conditional_edges(
    "evaluar_escalado",
    decidir_ruta,
    {"escalar": "escalar_a_humano", "enviar": "respuesta_automatica"}
)
builder.add_edge("escalar_a_humano", END)
builder.add_edge("respuesta_automatica", END)

agent = builder.compile()
print("✓ Agente compilado")

try:
    from IPython.display import display, Image
    display(Image(agent.get_graph().draw_mermaid_png()))
except Exception:
    print(agent.get_graph().draw_mermaid())

## Solución Ejercicio 4: Tests

In [ ]:
tickets_prueba = [
    "Compré unos auriculares hace dos semanas y están completamente rotos. Quiero mi dinero de vuelta YA.",
    "¿El altavoz Bluetooth modelo X es compatible con iOS 17?",
]

for ticket in tickets_prueba:
    print(f"\n{'='*60}\nTICKET: {ticket}\n{'='*60}")
    estado_inicial: TicketState = {
        "ticket": ticket, "category": "", "urgency": "", "summary": "",
        "rag_context": "", "draft_response": "",
        "escalate": False, "final_response": ""
    }
    resultado = agent.invoke(estado_inicial)
    print(f"\n{resultado['final_response']}")

## Solución Ejercicio 5 (bonus): Escalado con detección de riesgo legal

In [ ]:
PALABRAS_RIESGO_LEGAL = {"abogado", "denuncia", "consumidor", "juzgado", "demanda", "ilegal"}

def evaluar_escalado_v2(state: TicketState) -> dict:
    categorias_criticas = {"devolucion", "queja"}

    urgencia_critica = (
        state.get("urgency") == "alta"
        and state.get("category") in categorias_criticas
    )

    ticket_lower = state["ticket"].lower()
    riesgo_legal = any(palabra in ticket_lower for palabra in PALABRAS_RIESGO_LEGAL)

    escalar = urgencia_critica or riesgo_legal
    razon = []
    if urgencia_critica:
        razon.append(f"urgencia {state.get('urgency')} + categoría {state.get('category')}")
    if riesgo_legal:
        razon.append("riesgo legal detectado")

    print(f"  [evaluar_escalado_v2] → escalar: {escalar} ({', '.join(razon) if razon else 'sin motivo'})")
    return {"escalate": escalar}

# Probar con un ticket que menciona al abogado
ticket_legal = "Este producto es un fraude. Voy a hablar con mi abogado si no me devolvéis el dinero hoy."

builder_v2 = StateGraph(TicketState)
builder_v2.add_node("clasificar", clasificar)
builder_v2.add_node("consultar_catalogo", consultar_catalogo)
builder_v2.add_node("redactar_respuesta", redactar_respuesta)
builder_v2.add_node("evaluar_escalado", evaluar_escalado_v2)
builder_v2.add_node("respuesta_automatica", respuesta_automatica)
builder_v2.add_node("escalar_a_humano", escalar_a_humano)
builder_v2.set_entry_point("clasificar")
builder_v2.add_edge("clasificar", "consultar_catalogo")
builder_v2.add_edge("consultar_catalogo", "redactar_respuesta")
builder_v2.add_edge("redactar_respuesta", "evaluar_escalado")
builder_v2.add_conditional_edges(
    "evaluar_escalado", decidir_ruta,
    {"escalar": "escalar_a_humano", "enviar": "respuesta_automatica"}
)
builder_v2.add_edge("escalar_a_humano", END)
builder_v2.add_edge("respuesta_automatica", END)

agent_v2 = builder_v2.compile()

estado_legal: TicketState = {
    "ticket": ticket_legal, "category": "", "urgency": "", "summary": "",
    "rag_context": "", "draft_response": "", "escalate": False, "final_response": ""
}
resultado_legal = agent_v2.invoke(estado_legal)
print(f"\n{resultado_legal['final_response']}")

---

## Parte 2: Ejemplo de agente diseñado — Gestión de Candidaturas (RRHH)

*(Ejemplo de referencia para el instructor)*

### El proceso elegido

**Nombre del proceso:** Screening inicial de candidaturas para ofertas de empleo

**Sector:** Recursos Humanos / empresa de servicios profesionales

**Problema que resuelve:** El equipo de RRHH recibe 200-500 CVs por oferta. El screening manual tarda 2-3 días y tiene inconsistencias entre evaluadores. El agente realiza un primer filtrado objetivo y genera un resumen estructurado para el recruiter.

### Estado del agente

```python
class CandidaturaState(TypedDict):
    cv_texto: str           # texto del CV extraído
    oferta_texto: str       # descripción del puesto
    requisitos_minimos: list[str]  # extraídos de la oferta
    match_score: float      # 0.0 - 1.0
    gaps: list[str]         # requisitos que no cumple
    fortalezas: list[str]   # puntos fuertes del candidato
    decision: str           # "avanzar" | "descartar" | "revisar_humano"
    resumen: str            # para el recruiter
```

### Nodos del grafo

| Nodo | Qué hace | Tipo |
|------|----------|------|
| `extraer_requisitos` | Extrae requisitos mínimos y deseables de la oferta | LLM |
| `analizar_cv` | Compara el CV con los requisitos y calcula match + gaps | LLM |
| `consultar_similares` | Recupera CVs de candidatos históricos exitosos (RAG) | RAG |
| `generar_resumen` | Redacta el resumen para el recruiter | LLM |
| `decidir_accion` | Decide avanzar / descartar / escalar | Regla + LLM |

### Gobernanza

**Acciones con efectos reales:** envío de email de descarte automático (solo si `match_score < 0.3` y sin señales de perfil atípico interesante)

**Límites de acción:** máximo 50 descartes automáticos/día; ningún descarte si el candidato proviene de referido interno

**Qué loggar:** score, gaps detectados, documentos RAG usados como referencia, decisión final y si fue override humano

**Riesgo EU AI Act:** sistemas de selección de empleo son alto riesgo → requieren auditoría, transparencia al candidato y posibilidad de recurso humano.